What was never bought: valuing food from own production
=======================================================

**Author:** Ethan Ligon



## What this is



Session 2 claimed that much of the food in an LSMS household was never
bought, put a number on it for Ghana, and moved on.  This notebook is where
you check that number, find out what it rests on, and see whether it holds
anywhere else.

The arc: take the obvious call, notice it answers a different question than
the one you asked, find the missing piece in the raw table, put it back, and
then ask how much you should trust the result.

-   **Prerequisites:** `lsms_library` on the release kernel, and access to the
    GhanaLSS microdata.  Sections 1 through 4 are Ghana only; section 5 asks
    you to pick your own country.



## Setup



In [1]:
%xmode Plain

import lsms_library as ll
import numpy as np, pandas as pd

ghana = ll.Country('GhanaLSS')
acq = ghana.food_acquired()
acq.head()

The index is $(i, t, v, j, u, s, \mathit{visit})$: household, wave,
cluster, item, unit, source, visit.  The source level `s` is the one this
notebook is about.



## 1.  The obvious call, and what it leaves out



Start where session 2 started.



In [1]:
food = ghana.food_expenditures()
food_total = food.groupby(['t', 'i']).sum().squeeze()
food_total.groupby('t').describe().round(0)

Now ask what sources that table actually contains, and compare it against
the raw acquisition table it was derived from.



In [1]:
print("food_expenditures sources:", food.index.get_level_values('s').unique().tolist())
print("food_acquired    sources:", acq.index.get_level_values('s').unique().tolist())

**Exercise 1.1.** `food_expenditures()` takes a `basis` argument.  Try
`basis`'total'=.  Does it recover the own-production rows for 2016-17?  For
1991-92?  Explain the difference before reading section 2.



## 1.  Why own production disappears



The two sources are not recorded the same way.



In [1]:
acq.groupby(['t', 's'])[['Quantity', 'Expenditure', 'Price']].count()

As the survey records them, own-production rows carry a `Quantity` and a
`Price` but no `Expenditure` from 1991-92 onward, while purchases carry a
`Quantity` and an `Expenditure` but no `Price`.  In the two 1980s waves the
convention is the other way round for own production.  The survey answered
one question in two different currencies.

Since `lsms_library` 0.14.0 the library closes most of that gap itself: it
multiplies quantity by the household's own reported price and serves the
result as `Expenditure`, naming the rule in the `Derivation` column.  Read
the `Expenditure` column of the table above and you will find exactly one
wave still empty for `produced` — 1991-92, whose dominant unit is `All`,
"the whole harvest", which is not a quantity anything can be multiplied by.
So anything that sums `Expenditure` now drops own production for that one
wave rather than for six.



## 1.  Putting it back



The valuation Deaton and Zaidi devote a chapter to is, here, one
multiplication.



In [1]:
value = acq.Expenditure.where(acq.Expenditure.notna(), acq.Quantity * acq.Price)
by_source = value.groupby(['t', 's']).sum().unstack('s')
own_share = (100 * by_source['produced'] / by_source.sum(axis=1)).round(1)
own_share

Computed on 2026-09-17 this runs 38.8, 33.4, 26.7, 30.5, 17.1, 27.6, 28.7
across the seven waves.  Deaton and Zaidi's Table 3.1 puts home production
at 20.8% of the whole consumption aggregate for Ghana and food at 65.2% of
it, implying 31.9% of food, which is close to the modern waves here and was
arrived at by a different route.

**Exercise 3.1.** 2005-06 comes in at 17.1%, well under half its neighbours on
either side.  Find out why.  Start by asking whether the anomaly is in the
numerator or the denominator, and whether it survives restricting to
households that appear in both 2005-06 and 2012-13.

**Exercise 3.2.** The share above is of *food*.  Session 2's aside quotes
shares of the whole consumption aggregate.  What else would you need to
convert one into the other, and is it in this survey?



## 1.  How much should you trust it?



Two checks, and they point in different directions.

First: is the price real, or is it a cell constant somebody imputed
upstream?  If it were imputed it would not vary within an item and unit.



In [1]:
prod = acq.xs('2016-17', level='t').xs('produced', level='s')
n_distinct = prod.groupby(['j', 'u'])['Price'].nunique()
n_distinct.describe(percentiles=[.25, .5, .75, .9]).round(2)

Second: households report a price for what they grew; other households
report expenditure and quantity for the same item bought in a market.  Those
two should agree, at least in the middle.



In [1]:
pu = acq.xs('2016-17', level='t').xs('purchased', level='s')
uv = (pu.Expenditure / pu.Quantity).replace([np.inf, -np.inf], np.nan).dropna()

cmp = pd.concat([uv.groupby(['j', 'u']).median().rename('purchased_unit_value'),
                 prod.Price.groupby(['j', 'u']).median().rename('produced_price')],
                axis=1).dropna()
cmp['ratio'] = cmp.produced_price / cmp.purchased_unit_value
cmp.ratio.describe(percentiles=[.1, .25, .5, .75, .9]).round(3)

The median ratio across 178 item-unit cells is 1.00.  The tenth and
ninetieth percentiles are 0.5 and 4.0.  Read those two facts together: the
aggregate is sound and any individual household's imputed value may not be,
which is exactly the situation in which you report the aggregate and refuse
to rank two households on it.

**Exercise 4.1.** The 28.7% is a floor.  Count the own-production rows in
2016-17 carrying a price but no quantity, and bound how much they could add
if each were valued at its item-unit median quantity.

**Exercise 4.2.** Redo section 3 valuing own production at the *median
purchased unit value* in the household's own cluster, falling back to
district, region, then national when the cell is empty.  This is the
hierarchy session 2 described.  Report how far up you had to climb, and how
much the answer moves.



## 1.  Does it hold anywhere else?



`ll.Country` will give you any country in the library.  Session 2's aside
reproduces Deaton and Zaidi's Table 3.1, which ranges from 35.2% of the
aggregate in Nepal to 2.2% in South Africa, so there is a lot of range to
find.



In [1]:
# Countries the library carries.  Pick one you care about.
ll.countries()[:40]

**Exercise 5.1.** Repeat sections 1 through 3 for a country of your choosing.
Two things to establish before you report a number: which sources its
`food_acquired` distinguishes, and which of `Quantity`, `Expenditure` and
`Price` each source actually carries.  Do not assume Ghana's pattern.

**Exercise 5.2.** GhanaLSS distinguishes two sources.  Find a country whose
survey separates gifts, or food eaten away from home, and say what that does
to the "never bought" share.  Then say what Ghana's number is missing.



## Exercises



1.  Sections 1 to 4 for GLSS6 (`2012-13`) rather than GLSS7.  Does anything
    about the recording convention change between the two rounds?
2.  Session 2 computes poverty and inequality on `food_total`, which omits
    own production entirely.  Recompute the weighted headcount for 2016-17
    on a food aggregate that includes it.  Does the poverty ranking of
    Ghana's regions change?  This is the question the omission actually
    costs you, and it is not obvious in advance which way it goes.
3.  Argue the other side.  Own production enters the aggregate only through
    an imputation, and session 2's aside notes that imputed values are less
    variable than the truth, so including them tends to understate
    inequality.  Quantify that here: compare the Gini of purchased food
    against the Gini of purchased-plus-imputed food, and say which number
    you would publish and why.

